# EDA — corpus sintético de correos
Juan Camilo Gonzales y Juan Jose Solarte. Los 128 ejemplos fueron redactados para el proyecto con asistencia de IA, sin correos reales ni fuentes externas. Es una muestra pequeña y balanceada por diseño, no Big Data en volumen ni evidencia de generalización corporativa. El análisis reproduce prácticas de calidad de datos sin añadir Spark u otra infraestructura innecesaria.

Se conservan los splits versionados: 80/24/24. La limpieza usa `prepare_text`; no modifica los originales. Las gráficas léxicas/de longitud se restringen a train y validation. Solo se comprueba el tamaño del test, no se seleccionan parámetros con él.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ai_service' / 'train.py').exists())
if str(root) not in sys.path: sys.path.insert(0, str(root))
from ai_service.academic import dataset_frame
dataset = root / 'ai_service/data/correos_sinteticos_v0.3.jsonl'
frame, quality = dataset_frame(dataset)
quality

In [ ]:
import pandas as pd
counts = pd.crosstab(frame['label'], frame['split'])
assert counts.sum().to_dict() == {'test': 24, 'train': 80, 'validation': 24}
assert quality['missing'] == quality['duplicateIds'] == quality['duplicateTexts'] == 0
counts

In [ ]:
import matplotlib.pyplot as plt
import os, tempfile
output = Path(os.environ.get('NLP_NOTEBOOK_OUTPUT', tempfile.mkdtemp(prefix='nlp-eda-')))
output.mkdir(parents=True, exist_ok=True)
exploratory = frame[frame['split'].isin(['train', 'validation'])]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pd.crosstab(exploratory['label'], exploratory['split']).plot.bar(ax=axes[0], title='Balance: train / validation')
axes[0].tick_params(axis='x', labelrotation=30)
exploratory.groupby('split')['characters'].plot.hist(ax=axes[1], alpha=0.5, legend=True)
axes[1].set_title('Longitud normalizada (caracteres)')
plt.tight_layout()
fig.savefig(output / 'eda_balance_longitud.png', dpi=150, bbox_inches='tight')
plt.show()
exploratory.groupby('label')['characters'].agg(['count', 'min', 'median', 'max'])

## Riesgos y decisiones
No hay duplicados exactos normalizados, pero eso no descarta plantillas o similitudes semánticas entre particiones. El etiquetado sintético y el tamaño reducido favorecen resultados optimistas; no representan prevalencia real, hilos, ruido u otros idiomas. Un corpus ampliado debe agrupar hilos/plantillas antes de dividir, documentar licencia y anonimización, y reservar un test nuevo. No se cambia este corpus ni se usa su test para mejorar el modelo.